In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("wb_sec")

!wandb login --relogin $secret_value_0


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [2]:
!pip install git+https://github.com/csebuetnlp/normalizer -q


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, EarlyStoppingCallback

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score


In [4]:
import shutil
shutil.copytree('/kaggle/input/notebooks/hasnatabdullah/fork-of-banglat5-training/bnmc_hf_dataset', '/kaggle/working/bnmc_hf_dataset')

'/kaggle/working/bnmc_hf_dataset'

In [5]:
from datasets import load_from_disk
ds = load_from_disk('/kaggle/working/bnmc_hf_dataset')

In [6]:
def input_processing(example):
    return {'input': f'multilabel classification: {example['text']}'}

ds = ds.map(input_processing)

In [7]:
from normalizer import normalize


tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglat5")

def tokenize_fn(batch):
    inputs = [normalize(text) for text in batch["input"]]
    targets = [normalize(text) for text in batch["labels"]]

    model_inputs = tokenizer(
        inputs, 
        truncation=True, 
        padding="max_length", 
        max_length=512
    )

    labels = tokenizer(
        text_target=targets, 
        truncation=True, 
        padding="max_length", 
        max_length=64
    )

    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] 
        for label in labels["input_ids"]
    ]

    return model_inputs


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [8]:
tokenized_dataset = ds.map(
    tokenize_fn,
    batched=True,
)

final_dataset = tokenized_dataset.remove_columns(['text', 'input'])

Map:   0%|          | 0/36144 [00:00<?, ? examples/s]

Map:   0%|          | 0/4519 [00:00<?, ? examples/s]

Map:   0%|          | 0/4518 [00:00<?, ? examples/s]

In [9]:
df_2 = pd.read_csv('/kaggle/input/notebooks/hasnatabdullah/dataset-creation/bnmc_dataset.csv')

label_flat = df_2['labels'].str.split(', ').explode().unique()
all_labels = list(label_flat)

mlb = MultiLabelBinarizer(classes=all_labels)

mlb.fit(all_labels) 
len(mlb.classes_)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    preds = preds[0] if isinstance(preds, tuple) else preds
    preds = np.where(preds >= 0, preds, tokenizer.pad_token_id)
    labels = np.where(labels >= 0, labels, tokenizer.pad_token_id)
    decoder_preds = tokenizer.batch_decode(preds, skip_special_tokens =True)
    decoder_labels = tokenizer.batch_decode(labels, skip_special_tokens = True)
    pred_lists = [[item.strip() for item in text.split(',')] for text in decoder_preds]
    label_lists = [[item.strip() for item in text.split(',')] for text in decoder_labels]
    y_pred = mlb.transform(pred_lists)
    y_true = mlb.transform(label_lists)
    return{
        'f1_micro': f1_score(y_true, y_pred, average='micro'),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'hamming_loss': np.sum(np.not_equal(y_true, y_pred))/ float(y_true.size)
    }
    

In [10]:
model_path = "/kaggle/input/notebooks/hasnatabdullah/fork-of-banglat5-training/bnt5exp/checkpoint-20331"
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [11]:
training_args = Seq2SeqTrainingArguments(
    output_dir= 'bnt5exp_part_2',
    run_name = 'banglaT5_tuning_v3_p2',
    eval_strategy= 'epoch',
    save_strategy='epoch',
    learning_rate = 2e-5,
    warmup_steps=500,
    per_device_train_batch_size =4,
    gradient_accumulation_steps=2,
    num_train_epochs =18,
    predict_with_generate =True,
    generation_max_length=64,
    fp16 = True,
    logging_steps = 10,
    load_best_model_at_end = True,
    metric_for_best_model = 'f1_micro',
    greater_is_better = True,
    save_total_limit = 2,
    report_to = 'wandb'
)

In [12]:
early_stopping = EarlyStoppingCallback(early_stopping_patience=3)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100, padding=True)
trainer = Seq2SeqTrainer(
    model = model,
    args = training_args,
    train_dataset = final_dataset['train'],
    eval_dataset = final_dataset['val'],
    data_collator = data_collator,
    processing_class = tokenizer,
    compute_metrics = compute_metrics,
    callbacks = [early_stopping]
)

In [13]:
trainer.train(resume_from_checkpoint=model_path)

You are resuming training from a checkpoint trained with 4.57.1 of Transformers but your current version is 5.2.0. This is not recommended and could yield to errors or unwanted behaviors.
There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: hasnatabdullah47 (uolo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run mlmfihm1
wandb: Tracking run with wandb version 0.24.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260305_070528-mlmfihm1
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run banglaT5_tuning_v3_p2
wandb: ⭐️ View project at https://wandb.ai/uolo/huggingface
wandb: 🚀 View run at https://wandb.ai/uolo/huggingface/runs/mlmfihm1
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked 

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
10,1.569585,0.404303,0.846954,0.705082,0.025921,481.653700,9.380000,0.588000
11,1.863421,0.400657,0.848821,0.706887,0.025710,478.809900,9.436000,0.591000
12,1.938584,0.396074,0.849112,0.709845,0.025532,481.019300,9.393000,0.588000
13,1.653636,0.394701,0.850337,0.708379,0.025247,482.069900,9.372000,0.587000
14,1.794418,0.390445,0.850946,0.714566,0.025252,484.098400,9.333000,0.585000
15,1.479497,0.386369,0.852551,0.715787,0.025134,484.955300,9.316000,0.584000
16,1.749606,0.386449,0.851643,0.712617,0.025095,484.214600,9.331000,0.584000
17,1.491871,0.386050,0.852968,0.716917,0.024996,485.316500,9.309000,0.583000
18,1.728235,0.385648,0.852882,0.715700,0.024991,487.307800,9.271000,0.581000


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['আবহাওয়া', 'জামায়াত', 'শেয়ারবাজার'] will be ignored
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['আবহাওয়া', 'জামায়াত', 'শেয়ারবাজার'] will be ignored
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['আবহাওয়া', 'জামায়াত', 'শেয়ারবাজার'] will be ignored
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=40662, training_loss=0.8619328450713342, metrics={'train_runtime': 40467.3503, 'train_samples_per_second': 16.077, 'train_steps_per_second': 1.005, 'total_flos': 4.454976554438492e+17, 'train_loss': 0.8619328450713342, 'epoch': 18.0})

In [14]:
trainer.save_model("bangla_t5_final")
tokenizer.save_pretrained("bangla_t5_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bangla_t5_final/tokenizer_config.json', 'bangla_t5_final/tokenizer.json')